# Data Acquisition: Shape of the Raw Data

Goal: prove we pulled each dataset down and that it opens in a dataframe, and note the shape of what we got.

For each dataset:
1. **Load** it into a polars DataFrame
2. **Row / column counts**
3. **What a row represents**
4. **Key columns + types**
5. **Time range**
6. **Geography**

Raw files live in `data/raw/` (gitignored). To get them, run from the project root:

```
uv run python scripts/fetch_alpr_cameras.py --state Illinois --out data/raw/Illinois_alpr.geojson
uv run python scripts/fetch_eyesonflock.py
uv run python scripts/fetch_atlas.py
uv run python scripts/fetch_census.py
```

In [ ]:
import io
import json
import zipfile
from datetime import date
from pathlib import Path

import polars as pl

# Find data/raw/ whether the notebook runs from notebooks/ or the project root
#ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
#RAW = ROOT / "data" / "raw"

# Find the project root (the folder with pyproject.toml) by walking up from wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"


# Show every column and cut long text short when printing tables
# Docs: https://docs.pola.rs/api/python/stable/reference/api/polars.Config.set_tbl_cols.html
#       https://docs.pola.rs/api/python/stable/reference/api/polars.Config.set_fmt_str_lengths.html
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(40);  # ; hides the output

## 1. OSM ALPR Cameras (primary)

Source: OpenStreetMap, pulled with the Overpass API by `scripts/fetch_alpr_cameras.py`.
File: `data/raw/Illinois_alpr.geojson`

**One row = one ALPR camera.**

In [ ]:
# Load: the file is GeoJSON, one "feature" per camera. Each feature has
# geometry.coordinates ([lon, lat], longitude FIRST) and a dict of properties.
# Flatten each camera into one dict, then pl.DataFrame turns the list of dicts into a table.
# Docs: https://docs.python.org/3/library/json.html#json.load
#       https://docs.pola.rs/api/python/stable/reference/dataframe/index.html
with open(RAW / "Illinois_alpr.geojson", encoding="utf-8") as f:
    geojson = json.load(f)

rows = []
for feature in geojson["features"]:
    lon, lat = feature["geometry"]["coordinates"]
    rows.append({"lon": lon, "lat": lat, **feature["properties"]})

cams = pl.DataFrame(rows)

In [ ]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
cams.shape

In [ ]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
cams.schema

In [ ]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
cams.head()

In [ ]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
cams.null_count()

In [ ]:
# Time range: when cameras were LAST edited in OSM (ISO text, which sorts in date order).
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.min.html
cams.select(
    pl.col("timestamp").min().alias("first_edit"),
    pl.col("timestamp").max().alias("last_edit"),
)

In [ ]:
# Geography: the box that contains every camera.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.select.html
cams.select(
    pl.col("lat").min().alias("lat_min"),
    pl.col("lat").max().alias("lat_max"),
    pl.col("lon").min().alias("lon_min"),
    pl.col("lon").max().alias("lon_max"),
)

## 2. Eyes On Flock (primary)

Source: eyesonflock.com's data endpoint (`/api/v1/data`), saved by `scripts/fetch_eyesonflock.py`.
File: the newest `data/raw/eyesonflock_*.json`. It holds `summary` (national totals) and `portals` (one entry per agency); the table is `portals`.

**One row = one agency's Flock transparency portal.**

In [ ]:
# Load: pick the newest snapshot (file names end in YYYY-MM-DD, so sorted() puts it last),
# then turn the "portals" list into a DataFrame.
# Docs: https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob
#       https://docs.pola.rs/api/python/stable/reference/dataframe/index.html
eof_path = sorted(RAW.glob("eyesonflock_*.json"))[-1]
with open(eof_path, encoding="utf-8") as f:
    portals = pl.DataFrame(json.load(f)["portals"])

eof_path.name

In [ ]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
portals.shape

In [ ]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
portals.schema

In [ ]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
portals.head()

In [ ]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
portals.null_count()

In [ ]:
# Time range: when Eyes On Flock last updated each portal (ISO text, sorts in date order).
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.min.html
portals.select(
    pl.col("data_last_updated").min().alias("first_updated"),
    pl.col("data_last_updated").max().alias("last_updated"),
)

In [ ]:
# Geography: how many states, and how many agencies are city-level vs county-level.
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.n_unique.html
portals.select(
    pl.col("state").n_unique().alias("states"),
    pl.col("city").is_not_null().sum().alias("city_agencies"),
    pl.col("county").is_not_null().sum().alias("county_agencies"),
)

## 3. EFF Atlas of Surveillance (secondary)

Source: EFF's Atlas of Surveillance CSV export, saved by `scripts/fetch_atlas.py`.
File: the newest `data/raw/atlas_of_surveillance_*.csv`

**One row = one (agency, technology) pair**, e.g. "Rockford Police Department uses Gunshot Detection".

In [ ]:
# Load: pl.read_csv reads a CSV file into a DataFrame.
# Docs: https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html
atlas_path = sorted(RAW.glob("atlas_of_surveillance_*.csv"))[-1]
atlas = pl.read_csv(atlas_path)

atlas_path.name

In [ ]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
atlas.shape

In [ ]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
atlas.schema

In [ ]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
atlas.head()

In [ ]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
atlas.null_count()

`null_count()` shows 0 everywhere because this file stores blanks as empty text `""`, not as null.

In [ ]:
# Time range: "Link 1 Date" is the date of the evidence, as text like "10/06/2018" or "10/1/18".
# .str.to_date() turns text into a date: %Y = 4-digit year, %y = 2-digit year.
# pl.when().then().otherwise() picks the format; strict=False gives null for values that aren't dates.
# Some dates are typos (year 0005, 3023), so keep 1990 to today only.
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.to_date.html
#       https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.when.html
text = pl.col("Link 1 Date")
evidence_date = (
    pl.when(text.str.contains(r"^\d{1,2}/\d{1,2}/\d{2}$"))
    .then(text.str.to_date("%m/%d/%y", strict=False))
    .otherwise(text.str.to_date("%m/%d/%Y", strict=False))
)
atlas.select(evidence_date.alias("d")).filter(pl.col("d").is_between(date(1990, 1, 1), date.today())).select(
    pl.col("d").min().alias("earliest"),
    pl.col("d").max().alias("latest"),
    pl.len().alias("rows_with_a_date"),
)

In [ ]:
# Geography: how many different agencies and state/territory codes.
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.n_unique.html
atlas.select(
    pl.col("Agency").n_unique().alias("agencies"),
    pl.col("State").n_unique().alias("states"),
)

## 4. US Census by ZIP code (secondary)

Source: US Census Bureau, saved by `scripts/fetch_census.py` into `data/raw/census/`:
- `acsdt5y2024-b19013.dat`: ACS 5-year table B19013, median household income
- `acsdt5y2024-b01003.dat`: ACS 5-year table B01003, total population
- `2024_Gaz_zcta_national.zip`: 2024 Gazetteer, land area

The ACS files cover every kind of place (states, counties, tracts, ...); ZIP code rows are the ones whose `GEO_ID` starts with `860Z200US`.
`_E001` = the estimate, `_M001` = its margin of error.

**One row = one ZCTA** (ZIP Code Tabulation Area, the Census's version of a ZIP code).

In [ ]:
# Load the two ACS tables (separated by |) and keep only the ZIP code rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html
#       https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html
CENSUS = RAW / "census"
income_all = pl.read_csv(CENSUS / "acsdt5y2024-b19013.dat", separator="|")
population_all = pl.read_csv(CENSUS / "acsdt5y2024-b01003.dat", separator="|")

income = income_all.filter(pl.col("GEO_ID").str.starts_with("860Z200US"))
population = population_all.filter(pl.col("GEO_ID").str.starts_with("860Z200US"))

# Load the Gazetteer: a zip holding one tab-separated file.
# GEOID is read as text so ZIPs like "00601" keep their leading zeros.
# Docs: https://docs.python.org/3/library/zipfile.html#zipfile.ZipFile
with zipfile.ZipFile(CENSUS / "2024_Gaz_zcta_national.zip") as z:
    gazetteer = pl.read_csv(io.BytesIO(z.read(z.namelist()[0])), separator="\t", schema_overrides={"GEOID": pl.String})

# The last column name has trailing spaces ("INTPTLONG      "), so strip every column name.
gazetteer.columns = [col.strip() for col in gazetteer.columns]

# Rows before keeping only ZIP code rows (every kind of place)
{"income_all": income_all.shape, "population_all": population_all.shape}

**Income (B19013), ZIP rows**

In [ ]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
income.shape

In [ ]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
income.schema

In [ ]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
income.head()

In [ ]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
income.null_count()

**Population (B01003), ZIP rows**

In [ ]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
population.shape

In [ ]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
population.schema

In [ ]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
population.head()

In [ ]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
population.null_count()

**Gazetteer (land area)**

In [ ]:
# .shape gives (number of rows, number of columns).
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html
gazetteer.shape

In [ ]:
# .schema lists every column name and its data type.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html
gazetteer.schema

In [ ]:
# .head() shows the first 5 rows.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html
gazetteer.head()

In [ ]:
# .null_count() counts the missing (null) values in each column.
# Docs: https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html
gazetteer.null_count()

Time range: ACS 5-year estimates average **2020–2024**; Gazetteer land areas are for **2024**.

In [ ]:
# Geography: how many ZCTAs, and the lowest/highest ZIP (00601 is in Puerto Rico).
# Docs: https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.n_unique.html
gazetteer.select(
    pl.col("GEOID").n_unique().alias("zctas"),
    pl.col("GEOID").min().alias("lowest_zip"),
    pl.col("GEOID").max().alias("highest_zip"),
)